In [ ]:
# ── CLEANUP — run this first, every time ────────────────────────────
running = False

try:
    camera.unobserve_all()
    camera.stop()
    print('Camera observers cleared.')
except Exception:
    pass

try:
    robot.left_motor.value  = 0.0
    robot.right_motor.value = 0.0
    print('Robot motors zeroed.')
except Exception:
    pass

camera = None
robot  = None
print('Cleanup done.')

# ── IMPORTS ─────────────────────────────────────────────────────────
import cv2
import numpy as np
import time
import json
import base64
from urllib import request, error
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg
import traitlets

print('Imports OK')

In [ ]:
import yaml

# ── Roboflow API Settings ────────────────────────────────────────────
API_KEY = "Ub1KVwtGHHdLLKRzoxdG"
API_URL = "https://serverless.roboflow.com/kais-workspace-stbmo/workflows/detect-count-and-visualize-3"
CONFIDENCE_THRESHOLD = 0.8
TARGET_CLASS = "bottle cap"
INFERENCE_INTERVAL = 1.0 # Seconds between API calls to prevent lag

# ── Navigation Configuration ─────────────────────────────────────────
CAMERA_WIDTH  = 320
CAMERA_HEIGHT = 240
TAPE_HSV_LOWER = np.array([15,  60,  60])
TAPE_HSV_UPPER = np.array([45, 255, 255])

CAMERA_TILT_DOWN = 15   

FORWARD_SPEED = 0.18
REVERSE_SPEED = 0.15
TURN_SPEED    = 0.22
MAX_SPEED     = 0.25

AVOID_REVERSE_TIME  = 0.40
AVOID_TURN_TIME     = 0.55
WANDER_COOLDOWN     = 0.50
COLLECTING_COOLDOWN = 3.0 # How long the robot stops when it finds a cap

YELLOW_ROI_START = 0.65
FRONT_ROI_TOP_FRAC    = 0.15
FRONT_ROI_BOTTOM_FRAC = 0.60

YELLOW_BOUNDARY_THRESHOLD = 1800
OBSTACLE_EDGE_THRESHOLD   = 500
OBSTACLE_EQUAL_THRESH     = 50

CANNY_LOW  = 40
CANNY_HIGH = 120

print("Configuration loaded.")

In [ ]:
def clamp(val, lo, hi):
    if val < lo: return lo
    if val > hi: return hi
    return val

def safe_stop():
    try:
        robot.left_motor.value  = 0.0
        robot.right_motor.value = 0.0
    except Exception:
        pass

def set_drive(left, right):
    try:
        robot.left_motor.value  = clamp(left,  -MAX_SPEED, MAX_SPEED)
        robot.right_motor.value = clamp(right, -MAX_SPEED, MAX_SPEED)
    except Exception:
        pass

# ── ROBOFLOW INFERENCE ───────────────────────────────────────────────
def detect_bottle_cap(frame):
    """Sends frame to Roboflow and returns True if a bottle cap is found."""
    try:
        ok, enc = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
        if not ok: return False
        
        image_b64 = base64.b64encode(enc.tobytes()).decode("utf-8")
        payload = {
            "api_key": API_KEY,
            "inputs": {
                "image": {"type": "base64", "value": image_b64},
                "confidence": CONFIDENCE_THRESHOLD,
            },
        }
        
        req = request.Request(
            url=API_URL,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json", "Accept": "application/json"},
            method="POST",
        )
        
        with request.urlopen(req, timeout=3.0) as resp:
            result = json.loads(resp.read().decode("utf-8"))
            
        outputs = result.get("outputs", [])
        if outputs:
            preds = outputs[0].get("predictions", {})
            if isinstance(preds, dict):
                preds = preds.get("predictions", [])
            
            for p in preds:
                if p.get("class") == TARGET_CLASS:
                    return True
        return False
    except Exception as e:
        # Fails silently so it doesn't crash the driving loop
        return False

# ── LOCAL VISION (Keep your original detect_yellow_boundary and detect_obstacle here) ──
# (Paste your existing detect_yellow_boundary and detect_obstacle functions here)

In [ ]:
nav_state           = 'WANDER'
avoid_start_time    = 0.0
avoid_turn_right    = True
_obstacle_alt_right = True
wander_cooldown_end = 0.0
collect_start_time  = 0.0
last_inference_time = 0.0 # Throttle API calls
last_left_spd       = 0.0
last_right_spd      = 0.0
running             = False   

def execute(change):
    global nav_state, avoid_start_time, avoid_turn_right
    global _obstacle_alt_right, wander_cooldown_end
    global collect_start_time, last_inference_time
    global last_left_spd, last_right_spd

    if not running: return

    frame     = change['new']
    left_spd  = 0.0
    right_spd = 0.0
    now       = time.time()

    try:
        # 1. API Inference Throttle (Check for cap only once per second while wandering)
        cap_detected = False
        if nav_state == 'WANDER' and (now - last_inference_time > INFERENCE_INTERVAL):
            cap_detected = detect_bottle_cap(frame)
            last_inference_time = now

        # 2. Local Vision
        boundary = detect_yellow_boundary(frame) # Make sure this function is defined above!
        obstacle = detect_obstacle(frame)        # Make sure this function is defined above!

        # 3. State Logic
        if nav_state == 'STOPPED':
            pass

        elif nav_state == 'COLLECT':
            if now - collect_start_time < COLLECTING_COOLDOWN:
                left_spd  = 0.0
                right_spd = 0.0
            else:
                nav_state = 'WANDER'
                wander_cooldown_end = now + WANDER_COOLDOWN # Give it a sec before looking for trouble again

        elif nav_state in ('AVOID_YELLOW', 'AVOID_OBSTACLE'):
            elapsed = now - avoid_start_time
            if elapsed < AVOID_REVERSE_TIME:
                left_spd, right_spd = -REVERSE_SPEED, -REVERSE_SPEED
            elif elapsed < AVOID_REVERSE_TIME + AVOID_TURN_TIME:
                if avoid_turn_right: left_spd, right_spd = TURN_SPEED, -TURN_SPEED
                else:                left_spd, right_spd = -TURN_SPEED, TURN_SPEED
            else:
                wander_cooldown_end = now + WANDER_COOLDOWN
                nav_state = 'WANDER'
                left_spd, right_spd = FORWARD_SPEED, FORWARD_SPEED

        elif nav_state == 'WANDER':
            if cap_detected:
                nav_state = 'COLLECT'
                collect_start_time = now
                left_spd, right_spd = 0.0, 0.0
                print(">>> BOTTLE CAP DETECTED! STOPPING TO COLLECT! <<<")
            elif now < wander_cooldown_end:
                left_spd, right_spd = FORWARD_SPEED, FORWARD_SPEED
            elif boundary['detected']:
                avoid_turn_right = boundary['yellow_left'] >= boundary['yellow_right']
                avoid_start_time = now
                nav_state        = 'AVOID_YELLOW'
                left_spd, right_spd = -REVERSE_SPEED, -REVERSE_SPEED
            elif obstacle['detected']:
                el, er = obstacle['edge_left'], obstacle['edge_right']
                if el - er > OBSTACLE_EQUAL_THRESH:   avoid_turn_right = True
                elif er - el > OBSTACLE_EQUAL_THRESH: avoid_turn_right = False
                else:
                    avoid_turn_right = _obstacle_alt_right
                    _obstacle_alt_right = not _obstacle_alt_right
                avoid_start_time = now
                nav_state        = 'AVOID_OBSTACLE'
                left_spd, right_spd = -REVERSE_SPEED, -REVERSE_SPEED
            else:
                left_spd, right_spd = FORWARD_SPEED, FORWARD_SPEED

        # 4. Drive & Update UI
        set_drive(left_spd, right_spd)
        last_left_spd, last_right_spd = left_spd, right_spd

        try:
            state_label.value = 'State: ' + nav_state
            motor_label.value = f'L={left_spd:.2f}  R={right_spd:.2f}'
        except Exception:
            pass

    except Exception as e:
        safe_stop()
        print('[ERROR] ' + str(e))

print('execute() defined with COLLECT state.')